# IT7009 Artificial Intelligence — Practical Project# Intelligent Traffic Incident Analysis System**Objective:** build a machine-learning pipeline that (i) predicts the **severity** of a roadtraffic incident (4-class classification), (ii) discovers **high-risk locations** and naturalgroupings of traffic conditions through **unsupervised learning**, and (iii) performs rigorous**feature analysis** to justify the input variables used.**Workflow**1. Setup and reproducibility2. Data loading3. Data-quality assessment (missing, duplicates, dtypes, formatting, invalid values, outliers, imbalance)4. Cleaning and preprocessing (with justification)5. Exploratory data analysis and visualisation6. Feature engineering7. Leakage and redundancy analysis8. Feature analysis and selection (importance, mutual information, subset ablation)9. Supervised learning (>=3 classifiers, stratified pipeline, tuning, imbalance-aware evaluation)10. Unsupervised learning (k-means, elbow + silhouette, PCA, cluster profiling)11. Critical evaluation and reflection12. Conclusion and references

## 1. Setup and ReproducibilityAll libraries used are available by default in Google Colab. A single `RANDOM_STATE` is fixed soevery result in this notebook is reproducible. When the environment variable `FIG_DIR` is set thenotebook runs head-less and exports every figure to PNG (used to build the project report);otherwise figures are displayed inline.

In [ ]:
import os, warnings, jsonimport numpy as npimport pandas as pdimport matplotlibFIG_DIR = os.environ.get("FIG_DIR", "")          # when set -> head-less run that exports figuresif FIG_DIR:    matplotlib.use("Agg")    os.makedirs(FIG_DIR, exist_ok=True)import matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCVfrom sklearn.preprocessing import StandardScaler, OneHotEncoderfrom sklearn.compose import ColumnTransformerfrom sklearn.pipeline import Pipelinefrom sklearn.impute import SimpleImputerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifierfrom sklearn.neighbors import KNeighborsClassifierfrom sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,                             classification_report, confusion_matrix, ConfusionMatrixDisplay,                             silhouette_score)from sklearn.cluster import KMeans, AgglomerativeClusteringfrom sklearn.decomposition import PCAfrom sklearn.inspection import permutation_importancefrom sklearn.feature_selection import mutual_info_classifwarnings.filterwarnings("ignore")pd.set_option("display.max_columns", 100)pd.set_option("display.width", 200)RANDOM_STATE = 42np.random.seed(RANDOM_STATE)plt.rcParams.update({    "figure.figsize": (9, 5), "figure.dpi": 110, "axes.grid": True,    "grid.alpha": 0.3, "axes.titlesize": 12, "axes.titleweight": "bold", "font.size": 10,})sns.set_palette("viridis")print("Environment ready.  pandas", pd.__version__, "| numpy", np.__version__)def finish_fig(name=None, dpi=150):    """Finalise a figure: save it (head-less run) and/or display it (notebook)."""    plt.tight_layout()    if FIG_DIR and name:        plt.savefig(os.path.join(FIG_DIR, f"{name}.png"), dpi=dpi, bbox_inches="tight")    plt.show()RESULTS = {}   # collects metrics used to build the report tables

## 2. Data LoadingThe dataset is a single Excel workbook (`Dataset_Brief_1.xlsx`) with 50,010 incident records and72 columns. In Colab the file is uploaded interactively; locally the path can be supplied throughthe `DATA_PATH` environment variable.

In [ ]:
DATA_PATH = os.environ.get("DATA_PATH", "Dataset_Brief_1.xlsx")if not os.path.exists(DATA_PATH):    try:        from google.colab import files          # noqa: F401  (Colab only)        uploaded = files.upload()        DATA_PATH = list(uploaded.keys())[0]    except Exception as exc:                    # pragma: no cover        raise FileNotFoundError(            "Dataset not found. Upload 'Dataset_Brief_1.xlsx' or set the DATA_PATH variable."        ) from excdf_raw = pd.read_excel(DATA_PATH, sheet_name="Dataset_50000")print("Dataset loaded:", df_raw.shape[0], "rows x", df_raw.shape[1], "columns")df_raw.head(3)

In [ ]:
# Column inventory grouped by meaning -- this grouping drives the later feature analysis.COLUMN_GROUPS = {    "Identifier":            ["Incident_ID"],    "Temporal (raw)":        ["Start_Time", "End_Time"],    "Temporal (derived)":    ["Hour", "Day", "Month", "Year", "Weekday", "Weekend", "Peak_Hour", "Season"],    "Target":                ["Severity"],    "Incident":              ["Incident_Type"],    "Location":              ["State", "City", "District", "Latitude", "Longitude", "Postal_Code", "Road_Name"],    "Road":                  ["Road_Type", "Road_Surface", "Lane_Count", "Road_Width", "Speed_Limit"],    "Environment":           ["Weather_Condition", "Temperature", "Humidity", "Pressure", "Visibility",                              "Wind_Speed", "Precipitation", "Cloud_Cover", "Dew_Point", "UV_Index",                              "Light_Condition"],    "Traffic":               ["Traffic_Density", "Average_Speed", "Vehicle_Count", "Heavy_Vehicle_Percentage",                              "Pedestrian_Count", "Bicycle_Count", "Congestion_Index", "Vehicle_Type"],    "Infrastructure":        ["Traffic_Signal", "Junction", "Railway", "Crossing", "Stop_Sign", "Roundabout",                              "Amenity", "Station", "Bump", "Turning_Loop", "Give_Way", "No_Exit",                              "Camera_Available", "Sensor_Available", "Speed_Camera"],    "Proximity":             ["Nearby_School", "Nearby_Hospital", "Nearby_Market", "Nearby_Airport",                              "Nearby_Bus_Station"],    "Context":               ["Road_Work", "Holiday"],    "Outcome-like":          ["Emergency_Response_Time", "Response_Team", "Accident_History",                              "Travel_Time_Minutes", "Travel_Category", "Risk_Score"],}for g, cols in COLUMN_GROUPS.items():    missing = [c for c in cols if c not in df_raw.columns]    flag = f"   <-- MISSING {missing}" if missing else ""    print(f"{g:22s} ({len(cols):2d}): {', '.join(cols)}{flag}")

## 3. Data-Quality AssessmentBefore any modelling we audit the raw data for the issues listed in the brief: missing values,duplicates, inconsistent values, incorrect data types, formatting problems, invalid ranges andoutliers. Documenting these findings *first* justifies every cleaning decision made in Section 4.

### 3.1 Structure, data types and missing values

In [ ]:
quality = pd.DataFrame({    "dtype": df_raw.dtypes.astype(str),    "n_missing": df_raw.isna().sum(),    "pct_missing": (df_raw.isna().mean() * 100).round(3),    "n_unique": df_raw.nunique(dropna=True),})print("Total missing cells:", int(df_raw.isna().sum().sum()),      f"({df_raw.isna().mean().mean() * 100:.4f}% of all cells)")print("Columns containing at least one missing value:", int((quality.n_missing > 0).sum()), "of", df_raw.shape[1])print("\nColumns with missing values (top 12 by count):")print(quality[quality.n_missing > 0].sort_values("n_missing", ascending=False).head(12).to_string())print("\nNOTE: the TARGET column 'Severity' itself has",      int(df_raw['Severity'].isna().sum()), "missing values -> those rows must be dropped.")

In [ ]:
# Columns stored as text ('object') that clearly hold numbers -> incorrect data types.suspicious_text = []for c in df_raw.select_dtypes(include=["object", "string"]).columns:    coerced = pd.to_numeric(df_raw[c], errors="coerce")    ok = coerced.notna().sum()    if ok >= 0.95 * df_raw[c].notna().sum() and ok > 0:        suspicious_text.append((c, ok, df_raw[c].notna().sum()))print("Columns typed as text but holding >=95% numeric values (incorrect data type):")for c, ok, tot in suspicious_text:    print(f"  {c:26s} numeric-parsable {ok}/{tot}")print("\nThese must be coerced to numeric before scaling / correlation analysis.")

### 3.2 Duplicate records

In [ ]:
n_dup_rows = int(df_raw.duplicated().sum())n_dup_ids = int(df_raw["Incident_ID"].duplicated().sum())print("Exact duplicate rows          :", n_dup_rows)print("Duplicate Incident_ID values  :", n_dup_ids)print("\nExample of a duplicated incident (identical on every column):")if n_dup_rows:    dup_key = df_raw[df_raw.duplicated(keep=False)].sort_values("Incident_ID").iloc[0]["Incident_ID"]    print(df_raw[df_raw["Incident_ID"] == dup_key].T.head(14).to_string())print("\nDuplicates are data-entry artefacts and must be removed before modelling.")

### 3.3 Inconsistent categorical values (formatting)Several categorical columns contain the *same* category written with different casing or trailingwhite-space (e.g. `Collision` vs `collision `). Left untreated these inflate the number of dummyvariables and split the signal.

In [ ]:
cat_cols = ["Incident_Type", "Road_Type", "Road_Surface", "Weather_Condition", "Vehicle_Type",            "Light_Condition", "Response_Team", "Weekday", "Season", "Travel_Category", "State", "City"]for c in cat_cols:    vals = df_raw[c].dropna().astype(str)    dirty = sorted({v for v in vals.unique() if v != v.strip() or v != v.strip().title()})    if dirty:        print(f"{c:20s} -> non-canonical variants: {dirty}")

### 3.4 Temporal parsing and consistency

In [ ]:
st = pd.to_datetime(df_raw["Start_Time"], errors="coerce", format="mixed")et = pd.to_datetime(df_raw["End_Time"], errors="coerce", format="mixed")print("Start_Time unparsable/NaT :", int(st.isna().sum()))print("End_Time   unparsable/NaT :", int(et.isna().sum()))print("Records where End_Time < Start_Time (invalid duration):", int((et < st).sum()))print("Date range:", st.min(), "->", st.max())print("\nDerived temporal columns are consistent with Start_Time (sanity check):")for col, ref, label in [("Hour", st.dt.hour, "hour"), ("Day", st.dt.day, "day"),                        ("Month", st.dt.month, "month"), ("Year", st.dt.year, "year")]:    match = int((df_raw[col] == ref).sum())    print(f"  {col:6s} matches Start_Time.{label:5s}: {match}/{int(st.notna().sum())}")print("\n'Weekday' contains dirty labels (lower-case / abbreviated) that must be normalised:")print(df_raw["Weekday"].value_counts(dropna=False).to_string())

### 3.5 Invalid values outside the physical domain

In [ ]:
# Domain rules: the ranges each measurement must satisfy.DOMAIN_RULES = {    "Latitude":      (0, 90),      "Longitude":     (-180, 180),    "Humidity":      (0, 100),     "Visibility":    (0, 10),    "Precipitation": (0, 100),     "Cloud_Cover":   (0, 100),    "UV_Index":      (0, 11),      "Lane_Count":    (1, 20),    "Speed_Limit":   (5, 200),     "Heavy_Vehicle_Percentage": (0, 100),}numeric_view = df_raw.copy()for c in [c for c in df_raw.columns if df_raw[c].dtype == object]:    numeric_view[c] = pd.to_numeric(numeric_view[c], errors="coerce")rows = []for col, (lo, hi) in DOMAIN_RULES.items():    if col in numeric_view.columns:        s = numeric_view[col]        n_bad = int(((s < lo) | (s > hi)).sum())        if n_bad:            rows.append((col, lo, hi, n_bad, s.min(), s.max()))invalid = pd.DataFrame(rows, columns=["column", "valid_min", "valid_max", "n_invalid", "observed_min", "observed_max"])print("Values violating physical/domain constraints:")print(invalid.to_string(index=False) if len(invalid) else "  (none)")print("\nThese are impossible measurements (e.g. humidity of 150%, latitude of 95) and are")print("replaced by NaN in the cleaning stage so they can be imputed rather than trusted.")

### 3.6 Outliers (IQR rule)

In [ ]:
num_only = numeric_view.select_dtypes(include=[np.number]).drop(columns=["Severity"], errors="ignore")out_rows = []for c in num_only.columns:    s = num_only[c].dropna()    q1, q3 = s.quantile([0.25, 0.75])    iqr = q3 - q1    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr    n_out = int(((s < lo) | (s > hi)).sum())    if n_out:        out_rows.append((c, n_out, round(100 * n_out / len(s), 2), round(lo, 2), round(hi, 2)))outliers = pd.DataFrame(out_rows, columns=["column", "n_outliers", "pct", "lower_fence", "upper_fence"])outliers = outliers.sort_values("n_outliers", ascending=False)print("Outliers detected by the 1.5 x IQR rule:")print(outliers.to_string(index=False))

In [ ]:
# Visualise the spread of the most affected variables.top_out = outliers.head(6)["column"].tolist()fig, axes = plt.subplots(2, 3, figsize=(15, 8))for ax, col in zip(axes.ravel(), top_out):    sns.boxplot(y=num_only[col].dropna(), ax=ax, color="#4C72B0")    ax.set_title(col)    ax.set_ylabel("")fig.suptitle("Figure 1. Outlier inspection of the most affected numeric variables (IQR rule)",             fontsize=12, fontweight="bold")finish_fig("fig01_outliers")

### 3.7 Target class imbalanceThe severity label is highly imbalanced. This is a *critical* property: a trivial model that alwayspredicts the majority class (Severity 4) would already score ~55% accuracy while completely failingon the minority classes. Accuracy alone is therefore meaningless here and **macro-averaged F1** isadopted as the primary metric throughout.

In [ ]:
sev_counts = df_raw["Severity"].value_counts().sort_index()sev_pct = (df_raw["Severity"].value_counts(normalize=True).sort_index() * 100).round(2)imbalance = pd.DataFrame({"count": sev_counts, "percentage": sev_pct})print(imbalance.to_string())print("\nImbalance ratio (majority / minority): "      f"{sev_counts.max() / sev_counts.min():.1f} : 1")print("Majority-class (Severity 4) accuracy baseline:",      f"{sev_pct.loc[4.0]:.2f}%  <- this is why accuracy is misleading")RESULTS["severity_distribution"] = {int(k): int(v) for k, v in sev_counts.items()}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))sns.countplot(x="Severity", data=df_raw, ax=axes[0], palette="viridis")axes[0].set_title("(a) Incident count by severity class")axes[0].set_xlabel("Severity level"); axes[0].set_ylabel("Number of incidents")for p in axes[0].patches:    axes[0].annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2, p.get_height()),                     ha="center", va="bottom", fontsize=9)axes[1].pie(sev_counts, labels=[f"Severity {int(i)}" for i in sev_counts.index],            autopct="%1.1f%%", startangle=140, colors=sns.color_palette("viridis", 4),            wedgeprops={"edgecolor": "white"})axes[1].set_title("(b) Class share (%)")axes[1].grid(False)fig.suptitle("Figure 2. Severity class distribution (class imbalance)", fontsize=12, fontweight="bold")finish_fig("fig02_severity_distribution")

## 4. Cleaning and PreprocessingEvery transformation below addresses a specific issue identified in Section 3. The table summarisesthe **issue -> treatment -> justification** chain.| # | Issue found | Treatment | Justification ||---|-------------|-----------|---------------|| 1 | 14 exact duplicate rows | drop duplicates | duplicated incidents would double-count and bias training || 2 | 7 rows with missing **Severity** (target) | drop rows | the label cannot be imputed; target must be observed || 3 | ~5–7 missing values per column (~0.01%) | median (numeric) / mode (categorical) imputation | negligible missingness; median/mode preserve the distribution and avoid losing rows || 4 | Text stored as numbers (`Temperature`, `Average_Speed`, `Vehicle_Count`, `Road_Width`) | `pd.to_numeric` | correct types are required for scaling, correlation and trees || 5 | Dirty categorical labels (`collision `, `DRY`, `RAIN`, `car`, `day light`, `Fri`) | strip + title-case normalisation | removes duplicate categories so dummy encoding is meaningful || 6 | Invalid values (lat 95, lon −200, humidity 150, visibility −5, precipitation −3) | set to NaN then impute | impossible measurements would distort scaling and models || 7 | 1 record with End_Time < Start_Time | drop record | internally inconsistent; duration is undefined || 8 | `Start_Time`/`End_Time` stored as text | parse to datetime, derive features | enables temporal feature engineering || 9 | Identifier columns (`Incident_ID`, `Postal_Code`, `Road_Name`) | drop | row identifiers / near-unique keys carry no generalisable signal || 10 | Severe class imbalance | stratified split + `class_weight="balanced"` | prevents the majority class dominating the loss || 11 | Potential data leakage (`Risk_Score`, `Emergency_Response_Time`, `End_Time`) | drop (Section 7) | not available at prediction time / derived from the target |**Ordering note:** the datetimes are parsed *before* the invalid-value and duration checks so that thetemporal inconsistencies can be detected in the same pass.

In [ ]:
def clean_incidents(raw: pd.DataFrame):    """Reproducible cleaning pipeline. Returns (clean_df, audit_log)."""    audit = []    df = raw.copy()    audit.append(("start", len(df), df.shape[1]))    # (1) exact duplicate records ------------------------------------------------    before = len(df)    df = df.drop_duplicates()    audit.append(("drop duplicate rows", before - len(df), df.shape[1]))    # (2) rows with a missing target --------------------------------------------    before = len(df)    df = df[df["Severity"].notna()].copy()    audit.append(("drop rows with missing Severity", before - len(df), df.shape[1]))    # (3) correct the data types of numeric-looking text columns ------------------    for c in ["Temperature", "Average_Speed", "Vehicle_Count", "Emergency_Response_Time", "Road_Width", "Risk_Score"]:        if c in df.columns:            df[c] = pd.to_numeric(df[c], errors="coerce")    # (4) normalise categorical labels -------------------------------------------    for c in cat_cols:        if c in df.columns:            df[c] = df[c].astype("string").str.strip().str.title()    # map abbreviations / synonyms onto canonical labels    df["Weekday"] = df["Weekday"].replace({"Fri": "Friday", "Mon": "Monday", "Sat": "Saturday",                                           "Sun": "Sunday", "Thu": "Thursday", "Tue": "Tuesday",                                           "Wed": "Wednesday"})    df["Light_Condition"] = df["Light_Condition"].replace({"Day Light": "Daylight"})    # (5) parse datetimes and drop internally inconsistent records ----------------    df["Start_Time"] = pd.to_datetime(df["Start_Time"], errors="coerce", format="mixed")    df["End_Time"] = pd.to_datetime(df["End_Time"], errors="coerce", format="mixed")    before = len(df)    df = df[~(df["End_Time"] < df["Start_Time"])]    audit.append(("drop records with End_Time < Start_Time", before - len(df), df.shape[1]))    # (6) enforce physical domain constraints (invalid -> NaN -> imputed later) ----    for col, (lo, hi) in DOMAIN_RULES.items():        if col in df.columns:            df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan    # (7) target to integer -------------------------------------------------------    df["Severity"] = df["Severity"].astype(int)    audit.append(("final", len(df), df.shape[1]))    return df, pd.DataFrame(audit, columns=["step", "rows_removed_or_remaining", "n_columns"])df_clean, audit_log = clean_incidents(df_raw)print("Cleaning audit trail:")print(audit_log.to_string(index=False))print("\nCleaned shape:", df_clean.shape)print("Rows removed in total:", len(df_raw) - len(df_clean))RESULTS["rows_raw"] = int(len(df_raw)); RESULTS["rows_clean"] = int(len(df_clean))

In [ ]:
# Impute the small number of remaining missing values.num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()str_cols = [c for c in df_clean.columns if c not in num_cols]miss_before = int(df_clean.isna().sum().sum())for c in num_cols:    df_clean[c] = df_clean[c].fillna(df_clean[c].median())for c in str_cols:    mode = df_clean[c].mode(dropna=True)    if len(mode):        df_clean[c] = df_clean[c].fillna(mode.iloc[0])miss_after = int(df_clean.isna().sum().sum())print(f"Missing cells before imputation: {miss_before}")print(f"Missing cells after  imputation: {miss_after}")print("Numeric imputation -> median (robust to outliers); categorical -> mode (most frequent).")

In [ ]:
# Confirm the categorical labels are now canonical (issue 3.3 resolved).for c in ["Incident_Type", "Road_Surface", "Weather_Condition", "Vehicle_Type", "Light_Condition",          "Response_Team", "Weekday"]:    print(f"{c:20s}: {sorted(df_clean[c].dropna().unique().tolist())}")

## 5. Exploratory Data Analysis and VisualisationThe exploratory analysis answers three questions: how are the severity classes distributed (Section3.7), how are the individual features distributed, and which features show a relationship withseverity. Every figure below is referenced in the report.

### 5.1 Distributions of key numeric features

In [ ]:
key_numeric = ["Traffic_Density", "Vehicle_Count", "Accident_History", "Temperature", "Humidity",               "Visibility", "Wind_Speed", "Average_Speed", "Road_Width", "Travel_Time_Minutes",               "Congestion_Index", "Precipitation"]fig, axes = plt.subplots(3, 4, figsize=(18, 11))for ax, col in zip(axes.ravel(), key_numeric):    sns.histplot(df_clean[col].astype(float), bins=30, kde=True, ax=ax, color="#3B5BA5")    ax.set_title(col); ax.set_xlabel(""); ax.set_ylabel("")fig.suptitle("Figure 3. Distributions of key numeric traffic and environmental features",             fontsize=13, fontweight="bold", y=1.00)finish_fig("fig03_numeric_distributions")

### 5.2 Correlation structureThe heat-map reveals the linear association between numeric features and the target. Two groups ofvariables stand out: variables that are *highly* correlated with `Severity` (a leakage warning sign,investigated in Section 7) and pairs of features that are highly correlated with *each other*(redundancy).

In [ ]:
num_for_corr = df_clean.select_dtypes(include=[np.number])corr = num_for_corr.corr()target_corr = corr["Severity"].drop("Severity").sort_values(key=np.abs, ascending=False)fig, axes = plt.subplots(1, 2, figsize=(20, 8), gridspec_kw={"width_ratios": [1, 1.15]})top20 = target_corr.head(20)colors = ["#C44E52" if v > 0 else "#4C72B0" for v in top20.values]axes[0].barh(top20.index[::-1], top20.values[::-1], color=colors[::-1])axes[0].set_title("(a) Top 20 numeric features by |correlation| with Severity")axes[0].set_xlabel("Pearson correlation with Severity")sns.heatmap(corr.loc[top20.index, top20.index], cmap="coolwarm", center=0, ax=axes[1],            cbar_kws={"shrink": 0.8}, annot=False)axes[1].set_title("(b) Correlation heat-map of the same features")fig.suptitle("Figure 4. Correlation structure and association with the target", fontsize=13, fontweight="bold")finish_fig("fig04_correlation")print("Top 12 numeric features correlated with Severity:")print(target_corr.head(12).round(3).to_string())RESULTS["top_target_corr"] = {k: round(float(v), 3) for k, v in target_corr.head(12).items()}

### 5.3 Categorical features versus severityGrouping by the categorical variables shows which conditions are associated with more severeincidents. Bad weather (fog, storm, rain, snow) is visibly associated with higher severity than clearor cloudy conditions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))for ax, col in zip(axes.ravel(), ["Weather_Condition", "Light_Condition", "Road_Surface", "Incident_Type"]):    order = df_clean.groupby(col)["Severity"].mean().sort_values(ascending=False).index    sns.barplot(x=col, y="Severity", data=df_clean, order=order, ax=ax, palette="viridis",                errorbar="sd")    ax.set_title(f"Mean severity by {col}")    ax.tick_params(axis="x", rotation=25)    ax.set_ylabel("Mean severity"); ax.set_xlabel("")fig.suptitle("Figure 5. Mean incident severity across key categorical conditions",             fontsize=13, fontweight="bold")finish_fig("fig05_categorical_vs_severity")for col in ["Weather_Condition", "Light_Condition", "Road_Type", "Vehicle_Type"]:    g = df_clean.groupby(col)["Severity"].mean().sort_values(ascending=False).round(3)    print(f"\nMean severity by {col}:\n{g.to_string()}")

### 5.4 Relationship between traffic intensity and severity`Traffic_Density` is the strongest legitimate numeric signal in the dataset. Higher traffic density isassociated with progressively higher severity, which is consistent with congestion magnifying theconsequences of an incident.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))sns.boxplot(x="Severity", y="Traffic_Density", data=df_clean, ax=axes[0], palette="viridis")axes[0].set_title("(a) Traffic density by severity")sns.violinplot(x="Severity", y="Vehicle_Count", data=df_clean, ax=axes[1], palette="viridis")axes[1].set_title("(b) Vehicle count by severity")sns.boxplot(x="Severity", y="Accident_History", data=df_clean, ax=axes[2], palette="viridis")axes[2].set_title("(c) Accident history by severity")fig.suptitle("Figure 6. Traffic-intensity variables versus severity level", fontsize=13, fontweight="bold")finish_fig("fig06_traffic_vs_severity")

## 6. Feature EngineeringThree engineered features are added. Each one encodes domain knowledge that the raw columns expressonly implicitly, and all three are computable at the moment an incident is reported (no leakage):1. **`Incident_Duration_Min`** – minutes between `Start_Time` and `End_Time`. Represents how long the   road was affected. *Kept only for profiling/EDA — removed from the model because the end time is   only known after the incident is resolved (see Section 7).*2. **`Speed_Ratio`** – `Average_Speed / Speed_Limit`. Expresses the observed speed relative to the   legal limit; a ratio below 1 indicates congestion regardless of the absolute speed limit.3. **`Night_Incident`** – binary flag for incidents occurring outside daylight (Night/Dawn/Dusk).4. **`Adverse_Weather`** – binary flag for fog / rain / storm / snow, which Section 5.3 showed to be   associated with higher severity.5. **`Vehicle_Load`** – vehicles per lane (`Vehicle_Count / Lane_Count`), a density proxy that is   independent of road width.

In [ ]:
df_feat = df_clean.copy()# 1. incident duration (profiling only -- flagged as post-event information)df_feat["Incident_Duration_Min"] = (df_feat["End_Time"] - df_feat["Start_Time"]).dt.total_seconds() / 60# 2. speed ratiodf_feat["Speed_Ratio"] = (df_feat["Average_Speed"].astype(float) /                          df_feat["Speed_Limit"].astype(float).replace(0, np.nan))# 3. night-time flagdf_feat["Night_Incident"] = df_feat["Light_Condition"].isin(["Night", "Dawn", "Dusk"]).astype(int)# 4. adverse-weather flagdf_feat["Adverse_Weather"] = df_feat["Weather_Condition"].isin(["Fog", "Rain", "Storm", "Snow"]).astype(int)# 5. vehicle load per lanedf_feat["Vehicle_Load"] = (df_feat["Vehicle_Count"].astype(float) /                           df_feat["Lane_Count"].astype(float).replace(0, np.nan))df_feat[["Incident_Duration_Min", "Speed_Ratio", "Night_Incident", "Adverse_Weather", "Vehicle_Load"]] = \    df_feat[["Incident_Duration_Min", "Speed_Ratio", "Night_Incident", "Adverse_Weather", "Vehicle_Load"]].replace(        [np.inf, -np.inf], np.nan)print("Engineered feature summary:")print(df_feat[["Incident_Duration_Min", "Speed_Ratio", "Night_Incident",               "Adverse_Weather", "Vehicle_Load"]].describe().round(3).to_string())

In [ ]:
# Do the engineered flags actually separate the classes?print("Mean severity by engineered flag:")for c in ["Night_Incident", "Adverse_Weather"]:    print(f"\n{c}:")    print(df_feat.groupby(c)["Severity"].agg(["mean", "count"]).round(3).to_string())fig, axes = plt.subplots(1, 2, figsize=(14, 5))sns.barplot(x="Adverse_Weather", y="Severity", data=df_feat, ax=axes[0], palette="rocket",            errorbar="sd")axes[0].set_title("(a) Adverse weather vs mean severity")axes[0].set_xticklabels(["Favourable", "Adverse"])sns.violinplot(x="Severity", y="Speed_Ratio", data=df_feat, ax=axes[1], palette="viridis")axes[1].set_title("(b) Speed ratio by severity")fig.suptitle("Figure 7. Engineered features versus severity", fontsize=13, fontweight="bold")finish_fig("fig07_engineered_features")

## 7. Leakage and Redundancy AnalysisThis is the most important methodological step in the project. The brief explicitly asks for"prediction-time availability" and "potential for data leakage" to be considered. The table belowdocuments every candidate and the decision taken.| Feature | Evidence | Decision | Reason ||---|---|---|---|| `Risk_Score` | correlation **0.88** with Severity; class means 37.8 / 72.5 / 107.8 / 147.3 | **DROP** | monotonically derived from the target — a near-perfect proxy that would inflate scores artificially || `Emergency_Response_Time` | correlation 0.37; class means 8.3 / 14.7 / 20.5 / 27.6 | **DROP** | measured *after* the incident is reported, so it cannot exist at prediction time || `End_Time` / `Incident_Duration_Min` | available only when the incident is closed | **DROP** | post-event information || `Response_Team` | dispatched as a *consequence* of severity | **DROP** | outcome of the decision being predicted || `Incident_ID` | 50,000 near-unique values | **DROP** | row identifier || `Postal_Code`, `Road_Name` | 38k / 39k unique values | **DROP** | near-unique high-cardinality keys; cause over-fitting and no generalisation || `Average_Speed` ↔ `Speed_Limit` | correlation **0.89** | keep `Speed_Ratio`, drop `Average_Speed` | redundancy: the ratio is more informative and removes the collinear duplicate || `Dew_Point` ↔ `Temperature` | correlation **0.97** | **DROP** `Dew_Point` | near-duplicate of temperature; adds no independent information |

In [ ]:
# Quantify the leakage evidence before removing the columns.leak_candidates = ["Risk_Score", "Emergency_Response_Time", "Accident_History", "Vehicle_Count",                   "Travel_Time_Minutes", "Congestion_Index"]leak_rows = []for c in leak_candidates:    if c in df_feat.columns:        s = pd.to_numeric(df_feat[c], errors="coerce")        means = df_feat.assign(_v=s).groupby("Severity")["_v"].mean().round(2)        monotonic = bool(np.all(np.diff(means.values) > 0))        leak_rows.append((c, round(float(s.corr(df_feat["Severity"])), 3), monotonic,                          " / ".join(f"{v}" for v in means.values)))leak_table = pd.DataFrame(leak_rows, columns=["feature", "corr_with_Severity",                                              "class_means_monotonic", "class_means_1|2|3|4"])print("Leakage evidence:")print(leak_table.to_string(index=False))print("\nA feature whose class-means rise monotonically with the target AND correlates strongly with it")print("is almost certainly derived from the label -> leakage.")RESULTS["leakage_table"] = leak_table.to_dict("records")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))for ax, c in zip(axes, ["Risk_Score", "Emergency_Response_Time"]):    sns.boxplot(x="Severity", y=pd.to_numeric(df_feat[c], errors="coerce"), data=df_feat,                ax=ax, palette="rocket")    ax.set_title(f"{c} by severity  (corr = "                 f"{pd.to_numeric(df_feat[c], errors='coerce').corr(df_feat['Severity']):.2f})")    ax.set_ylabel(c)fig.suptitle("Figure 8. Data-leakage evidence: outcome variables track the target almost perfectly",             fontsize=13, fontweight="bold")finish_fig("fig08_leakage")

### 7.1 Redundancy

In [ ]:
redundancy_view = df_feat.select_dtypes(include=[np.number])corr_abs = redundancy_view.corr().abs()upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))redundant_pairs = [(c, r, round(float(upper.loc[r, c]), 3))                   for c in upper.columns for r in upper.index if pd.notna(upper.loc[r, c]) and upper.loc[r, c] > 0.8]print("Highly correlated (|r| > 0.8) feature pairs -> redundancy:")for a, b, v in redundant_pairs:    print(f"  {a}  <->  {b}   r = {v}")print("\nSpeed_Ratio is retained in place of Average_Speed because it normalises by Speed_Limit.")

In [ ]:
# Final feature selection -- apply the leakage / identifier / redundancy decisions.DROP_COLUMNS = [    # identifiers    "Incident_ID", "Postal_Code", "Road_Name",    # leakage / post-event information    "Risk_Score", "Emergency_Response_Time", "Response_Team", "End_Time", "Incident_Duration_Min",    # redundant    "Average_Speed", "Dew_Point",]model_df = df_feat.drop(columns=[c for c in DROP_COLUMNS if c in df_feat.columns]).copy()model_df = model_df.drop(columns=["Start_Time"])   # raw timestamp replaced by derived temporal featuresprint("Columns dropped:", [c for c in DROP_COLUMNS + ["Start_Time"] if c in df_feat.columns])print("Remaining feature matrix shape:", model_df.shape)print("Remaining columns:")print(list(model_df.columns))

## 8. Feature Analysis and SelectionThree complementary techniques are used to rank the remaining features:* **Mutual information** – captures non-linear dependence between each feature and the target.* **Random-forest Gini importance** – model-based ranking of predictive contribution.* **Permutation importance** – measures the drop in macro-F1 when a feature is shuffled; the most  reliable of the three because it is evaluated on held-out data with the actual metric.Finally, an **ablation study** measures how the macro-F1 of a fixed classifier changes as features areremoved, which directly satisfies the requirement to "assess the impact of different feature subsets".

In [ ]:
y_all = model_df["Severity"].astype(int)X_all = model_df.drop(columns=["Severity"])categorical_features = X_all.select_dtypes(include=["object", "string"]).columns.tolist()numeric_features = X_all.select_dtypes(include=[np.number]).columns.tolist()print("Categorical features (%d):" % len(categorical_features), categorical_features)print("Numeric features     (%d)" % len(numeric_features))

In [ ]:
# Build a shared preprocessing pipeline (impute -> scale numerics, one-hot categoricals).numeric_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])categorical_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),                             ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])preprocessor = ColumnTransformer([    ("num", numeric_pipe, numeric_features),    ("cat", categorical_pipe, categorical_features),])# Encode once for the feature-analysis routines.X_encoded = preprocessor.fit_transform(X_all)encoded_names = numeric_features + list(    preprocessor.named_transformers_["cat"]["onehot"].get_feature_names_out(categorical_features))print("Encoded feature matrix:", X_encoded.shape)

In [ ]:
# --- Mutual information -------------------------------------------------------mi_scores = mutual_info_classif(X_encoded, y_all, random_state=RANDOM_STATE)mi_series = pd.Series(mi_scores, index=encoded_names).sort_values(ascending=False)# --- Random-forest Gini importance -------------------------------------------rf_probe = RandomForestClassifier(n_estimators=250, random_state=RANDOM_STATE, n_jobs=-1,                                  class_weight="balanced")rf_probe.fit(X_encoded, y_all)gini_series = pd.Series(rf_probe.feature_importances_, index=encoded_names).sort_values(ascending=False)# --- Permutation importance on a hold-out split -------------------------------X_tr, X_te, y_tr, y_te = train_test_split(X_encoded, y_all, test_size=0.2,                                          random_state=RANDOM_STATE, stratify=y_all)rf_eval = RandomForestClassifier(n_estimators=250, random_state=RANDOM_STATE, n_jobs=-1,                                 class_weight="balanced").fit(X_tr, y_tr)perm = permutation_importance(rf_eval, X_te, y_te, n_repeats=5, random_state=RANDOM_STATE,                              scoring="f1_macro", n_jobs=-1)perm_series = pd.Series(perm.importances_mean, index=encoded_names).sort_values(ascending=False)importance_table = pd.DataFrame({    "mutual_information": mi_series,    "gini_importance": gini_series,    "permutation_importance": perm_series,})print("Top 20 features (average of the three rankings):")top_features = importance_table.mean(axis=1).sort_values(ascending=False).head(20)print(importance_table.loc[top_features.index].round(4).to_string())RESULTS["top_features"] = top_features.index.tolist()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))for ax, series, title in [(axes[0], mi_series, "(a) Mutual information"),                          (axes[1], gini_series, "(b) Random-forest Gini importance"),                          (axes[2], perm_series, "(c) Permutation importance (macro-F1)")]:    top = series.head(15)[::-1]    ax.barh(top.index, top.values, color="#55A868")    ax.set_title(title)    ax.tick_params(axis="y", labelsize=8)fig.suptitle("Figure 9. Feature-importance rankings from three independent methods",             fontsize=13, fontweight="bold")finish_fig("fig09_feature_importance")

### 8.1 Feature-subset ablation studyFour feature subsets are compared with a fixed classifier (random forest) so that the *only* change isthe input features:* **All features** – every remaining column.* **Top-20** – the 20 highest-ranked features from the importance analysis.* **Top-10** – the 10 highest-ranked features.* **No-leakage-guard** – all features *including* `Risk_Score` and `Emergency_Response_Time`, shown  only to demonstrate how leakage inflates the score (never used as a final model).

In [ ]:
def evaluate_subset(feature_names, label, source=None):    """Train a fixed RF on a feature subset and return macro-F1 / accuracy."""    src = model_df if source is None else source    cols = [c for c in feature_names if c in src.columns]    sub = src[cols + ["Severity"]].copy()    Xs = sub.drop(columns=["Severity"]); ys = sub["Severity"]    Xtr, Xte, ytr, yte = train_test_split(Xs, ys, test_size=0.2, random_state=RANDOM_STATE, stratify=ys)    num_c = Xs.select_dtypes(include=[np.number]).columns.tolist()    cat_c = [c for c in Xs.columns if c not in num_c]    pre = ColumnTransformer([        ("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), num_c),        ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")),                          ("o", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_c),    ])    pipe = Pipeline([("pre", pre), ("clf", RandomForestClassifier(        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced"))])    pipe.fit(Xtr, ytr)    pred = pipe.predict(Xte)    return {"subset": label, "n_features": Xs.shape[1],            "accuracy": round(accuracy_score(yte, pred), 4),            "macro_f1": round(f1_score(yte, pred, average="macro"), 4)}all_feature_cols = X_all.columns.tolist()top20_cols = [c for c in top_features.index[:20] if c in all_feature_cols]top10_cols = [c for c in top_features.index[:10] if c in all_feature_cols]ablation = pd.DataFrame([    evaluate_subset(all_feature_cols, "All features"),    evaluate_subset(top20_cols, "Top-20 features"),    evaluate_subset(top10_cols, "Top-10 features"),    evaluate_subset(all_feature_cols + ["Risk_Score", "Emergency_Response_Time"],                    "With leakage (Risk_Score + Response_Time)", source=df_feat),])print("Feature-subset ablation (random forest, identical settings):")print(ablation.to_string(index=False))RESULTS["ablation"] = ablation.to_dict("records")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))plot_df = ablation.set_index("subset")[["accuracy", "macro_f1"]]plot_df.plot(kind="bar", ax=ax, color=["#4C72B0", "#C44E52"], rot=15)ax.set_title("Figure 10. Feature-subset ablation study")ax.set_ylabel("Score"); ax.set_xlabel(""); ax.set_ylim(0, 1)for cont in ax.containers:    ax.bar_label(cont, fmt="%.3f", fontsize=8, padding=2)ax.legend(["Accuracy", "Macro F1"])finish_fig("fig10_ablation")

## 9. Supervised Learning — Severity Classification### 9.1 Design* **Task:** 4-class classification of incident severity.* **Split:** stratified 80 / 20 train–test split, so the rare Severity-1 class is represented in both  partitions in the same proportion. Stratification is essential given the 35:1 imbalance.* **Preprocessing:** median imputation + standardisation for numeric features; most-frequent  imputation + one-hot encoding for categoricals. All steps live inside a `Pipeline`, so they are  fitted on the training fold only — no information leaks from the test set.* **Algorithms (4, exceeding the required minimum of 3):**  1. **Logistic Regression** — linear, interpretable baseline; `class_weight="balanced"`.  2. **Random Forest** — bagged trees, handles non-linearity and mixed feature types.  3. **HistGradientBoosting** — boosted trees, typically the strongest tabular performer.  4. **k-Nearest Neighbours** — instance-based, distance-sensitive; included as a contrasting model.* **Primary metric:** macro-averaged F1 (accuracy, precision and recall are also reported).

In [ ]:
X = model_df.drop(columns=["Severity"])y = model_df["Severity"].astype(int)X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)print("Training set:", X_train.shape, " Test set:", X_test.shape)print("\nClass proportions are preserved by the stratified split:")print(pd.DataFrame({"train_%": (y_train.value_counts(normalize=True).sort_index() * 100).round(2),                    "test_%": (y_test.value_counts(normalize=True).sort_index() * 100).round(2)}).to_string())

In [ ]:
def make_pipeline(estimator, X_reference):    """Assemble preprocessing + estimator into a single leak-free pipeline."""    num_c = X_reference.select_dtypes(include=[np.number]).columns.tolist()    cat_c = [c for c in X_reference.columns if c not in num_c]    pre = ColumnTransformer([        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),                          ("scale", StandardScaler())]), num_c),        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_c),    ])    return Pipeline([("preprocess", pre), ("classifier", estimator)])# Reference point: what a "predict the majority class" model would score.majority = y_train.value_counts().idxmax()naive_pred = np.full_like(y_test, majority)print(f"Naive majority-class baseline (always predicts Severity {majority}):")print(f"  accuracy = {accuracy_score(y_test, naive_pred):.4f}   macro-F1 = {f1_score(y_test, naive_pred, average='macro'):.4f}")print("  -> a useful model must beat BOTH numbers, especially macro-F1.")RESULTS["naive"] = {"accuracy": round(accuracy_score(y_test, naive_pred), 4),                    "macro_f1": round(f1_score(y_test, naive_pred, average="macro"), 4)}

### 9.2 Baseline comparison of the four classifiers

In [ ]:
base_models = {    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced",                                              random_state=RANDOM_STATE),    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE,                                            n_jobs=-1, class_weight="balanced"),    "Hist Gradient Boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE,                                                             class_weight="balanced"),    "k-Nearest Neighbours": KNeighborsClassifier(n_neighbors=15, weights="distance", n_jobs=-1),}fitted, rows, preds = {}, [], {}for name, est in base_models.items():    pipe = make_pipeline(est, X_train)    pipe.fit(X_train, y_train)    y_pred = pipe.predict(X_test)    fitted[name] = pipe    preds[name] = y_pred    rows.append({        "model": name,        "accuracy": round(accuracy_score(y_test, y_pred), 4),        "precision_macro": round(precision_score(y_test, y_pred, average="macro", zero_division=0), 4),        "recall_macro": round(recall_score(y_test, y_pred, average="macro", zero_division=0), 4),        "f1_macro": round(f1_score(y_test, y_pred, average="macro"), 4),        "f1_weighted": round(f1_score(y_test, y_pred, average="weighted"), 4),    })baseline_results = pd.DataFrame(rows).sort_values("f1_macro", ascending=False).reset_index(drop=True)print("Baseline performance on the held-out test set:")print(baseline_results.to_string(index=False))RESULTS["baseline_results"] = baseline_results.to_dict("records")

In [ ]:
# Confusion matrices for every baseline model.fig, axes = plt.subplots(2, 2, figsize=(14, 12))for ax, name in zip(axes.ravel(), base_models.keys()):    cm = confusion_matrix(y_test, preds[name], labels=sorted(y.unique()))    ConfusionMatrixDisplay(cm, display_labels=sorted(y.unique())).plot(ax=ax, cmap="Blues", colorbar=False)    ax.set_title(f"{name}\nmacro-F1 = {f1_score(y_test, preds[name], average='macro'):.3f}")fig.suptitle("Figure 11. Confusion matrices of the four baseline classifiers (test set)",             fontsize=13, fontweight="bold")finish_fig("fig11_confusion_matrices_baseline")

### 9.3 Hyper-parameter tuningEach of the three strongest models is tuned with `GridSearchCV` using **3-fold stratifiedcross-validation** and **macro-F1** as the optimisation objective. Macro-F1 is used (rather thanaccuracy) precisely because the classes are imbalanced — it forces the search to care about the rareSeverity-1 class.

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)param_grids = {    "Logistic Regression": {        "classifier__C": [0.1, 1.0, 10.0],        "classifier__solver": ["lbfgs"],    },    "Random Forest": {        "classifier__n_estimators": [300],        "classifier__max_depth": [None, 24],        "classifier__min_samples_leaf": [1, 5],    },    "Hist Gradient Boosting": {        "classifier__learning_rate": [0.1],        "classifier__max_iter": [200, 400],        "classifier__max_leaf_nodes": [31, 63],    },}tuned_models, tuning_rows = {}, []for name, grid in param_grids.items():    search = GridSearchCV(make_pipeline(base_models[name], X_train), grid,                          scoring="f1_macro", cv=cv, n_jobs=-1, refit=True)    search.fit(X_train, y_train)    tuned_models[name] = search.best_estimator_    tuning_rows.append({"model": name,                        "best_cv_macro_f1": round(search.best_score_, 4),                        "best_params": {k.replace("classifier__", ""): v for k, v in search.best_params_.items()}})    print(f"{name}: best CV macro-F1 = {search.best_score_:.4f}  params = "          f"{ {k.replace('classifier__', ''): v for k, v in search.best_params_.items()} }")tuning_table = pd.DataFrame(tuning_rows)RESULTS["tuning"] = tuning_rows

### 9.4 Effect of class-imbalance handlingThe brief requires the imbalance to be addressed explicitly. Three strategies are compared on the samemodel and split:* **None** – the model optimises plain accuracy and is dominated by the majority class.* **`class_weight="balanced"`** – re-weights the loss inversely to class frequency.* **Random over-sampling** – duplicates minority-class training rows (equivalent to re-weighting but  implemented at the data level).

In [ ]:
def resample_train(X_tr, y_tr, random_state=RANDOM_STATE):    """Simple random over-sampling of the minority classes."""    parts = []    target = y_tr.value_counts().max()    for cls, group in X_tr.assign(_y=y_tr).groupby("_y"):        parts.append(group.sample(target, replace=True, random_state=random_state))    out = pd.concat(parts).sample(frac=1, random_state=random_state)    return out.drop(columns="_y"), out["_y"]imbalance_rows = []for strategy in ["None", "class_weight=balanced"]:    est = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE,                             class_weight=("balanced" if strategy == "class_weight=balanced" else None))    pipe = make_pipeline(est, X_train).fit(X_train, y_train)    pr = pipe.predict(X_test)    imbalance_rows.append({"strategy": strategy,                           "accuracy": round(accuracy_score(y_test, pr), 4),                           "macro_f1": round(f1_score(y_test, pr, average="macro"), 4),                           "minority_class_recall": round(recall_score(y_test, pr, labels=[1], average="macro", zero_division=0), 4)})X_over, y_over = resample_train(X_train, y_train)pipe_over = make_pipeline(LogisticRegression(max_iter=2000, random_state=RANDOM_STATE), X_train).fit(X_over, y_over)pr_over = pipe_over.predict(X_test)imbalance_rows.append({"strategy": "random over-sampling",                       "accuracy": round(accuracy_score(y_test, pr_over), 4),                       "macro_f1": round(f1_score(y_test, pr_over, average="macro"), 4),                       "minority_class_recall": round(recall_score(y_test, pr_over, labels=[1], average="macro", zero_division=0), 4)})imbalance_table = pd.DataFrame(imbalance_rows)print("Class-imbalance handling comparison (logistic regression):")print(imbalance_table.to_string(index=False))print("\nBalancing trades a small amount of accuracy for a large gain in macro-F1 and in the recall of")print("the rare Severity-1 incidents -- exactly the trade-off that matters operationally.")RESULTS["imbalance"] = imbalance_rows

### 9.5 Final evaluation of the tuned models

In [ ]:
final_rows, final_preds = [], {}for name, pipe in tuned_models.items():    pr = pipe.predict(X_test)    final_preds[name] = pr    final_rows.append({        "model": name,        "accuracy": round(accuracy_score(y_test, pr), 4),        "precision_macro": round(precision_score(y_test, pr, average="macro", zero_division=0), 4),        "recall_macro": round(recall_score(y_test, pr, average="macro", zero_division=0), 4),        "f1_macro": round(f1_score(y_test, pr, average="macro"), 4),        "f1_weighted": round(f1_score(y_test, pr, average="weighted"), 4),    })final_results = pd.DataFrame(final_rows).sort_values("f1_macro", ascending=False).reset_index(drop=True)print("Tuned model performance on the held-out test set:")print(final_results.to_string(index=False))RESULTS["final_results"] = final_results.to_dict("records")best_name = final_results.iloc[0]["model"]best_pipe = tuned_models[best_name]best_pred = final_preds[best_name]print(f"\nBest model by macro-F1: {best_name}")RESULTS["best_model"] = best_name

In [ ]:
# Detailed per-class report for the best model.print(f"Classification report -- {best_name}")print(classification_report(y_test, best_pred, digits=4))RESULTS["best_report"] = classification_report(y_test, best_pred, digits=4, output_dict=True)

### 9.6 Model comparison and confusion matrix of the best model

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6))metrics = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]final_results.set_index("model")[metrics].plot(kind="bar", ax=axes[0], rot=12,                                               color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"])axes[0].set_title("(a) Tuned models -- metric comparison")axes[0].set_ylim(0, 1); axes[0].set_ylabel("Score"); axes[0].set_xlabel("")axes[0].legend(fontsize=8, ncol=2)cm = confusion_matrix(y_test, best_pred, labels=sorted(y.unique()))ConfusionMatrixDisplay(cm, display_labels=sorted(y.unique())).plot(ax=axes[1], cmap="Greens", colorbar=False)axes[1].set_title(f"(b) Confusion matrix -- best model ({best_name})")fig.suptitle("Figure 12. Supervised model comparison and confusion matrix of the best model",             fontsize=13, fontweight="bold")finish_fig("fig12_model_comparison")

In [ ]:
# Per-class recall shows the operational behaviour: how many incidents of each severity are caught.per_class = pd.DataFrame(RESULTS["best_report"]).Tper_class = per_class.loc[[str(i) for i in sorted(y.unique())], ["precision", "recall", "f1-score", "support"]]print(f"Per-class performance -- {best_name}")print(per_class.round(4).to_string())RESULTS["per_class"] = per_class.round(4).to_dict("index")

In [ ]:
# Normalised confusion matrix highlights where the model confuses adjacent severities.cm_norm = confusion_matrix(y_test, best_pred, labels=sorted(y.unique()), normalize="true")fig, ax = plt.subplots(figsize=(7, 6))sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Greens", ax=ax,            xticklabels=sorted(y.unique()), yticklabels=sorted(y.unique()))ax.set_xlabel("Predicted severity"); ax.set_ylabel("Actual severity")ax.set_title("Figure 13. Normalised confusion matrix of the best model")finish_fig("fig13_best_confusion_matrix_normalized")

## 10. Unsupervised Learning — Discovering Traffic-Condition Patterns### 10.1 DesignUnsupervised learning is applied **without using the Severity label**. The objective is to discovernatural groupings of traffic conditions and to flag abnormal (rare) situations that may indicateemerging incidents or accident hotspots.* **Features:** the numeric attributes only (scaled with `StandardScaler`), because k-means and  silhouette are distance-based. Categorical variables are excluded from the distance metric but used  *afterwards* to profile the clusters.* **Algorithm:** k-means, tested for `k = 2 … 10`.* **Model selection:** the **elbow method** (inertia) and the **silhouette score** are used together,  as required by the brief. Silhouette is computed on a 10,000-point sample because it is O(n²).  Because the silhouette curve turns out to be flat and uniformly low, the elbow and the operational  interpretability of the solution are used as the deciding evidence — the trade-off is discussed  explicitly in Section 10.2.* **Validation:** the selected solution is visualised with PCA, profiled feature-by-feature, and  cross-tabulated against the (unused) Severity label to test whether the discovered structure has  operational meaning.* **Anomaly detection:** the distance of each incident from its cluster centroid is used to flag  abnormal conditions (the most distant 1% of records).

In [ ]:
cluster_features = numeric_features.copy()          # numeric attributes only (distance-based)if "Severity" in cluster_features:    cluster_features.remove("Severity")X_cluster = model_df[cluster_features].astype(float)X_cluster = X_cluster.fillna(X_cluster.median())scaler = StandardScaler()X_scaled = scaler.fit_transform(X_cluster)print("Clustering matrix:", X_scaled.shape, "->", len(cluster_features), "numeric features")

### 10.2 Choosing the number of clusters — elbow method and silhouette score

In [ ]:
k_range = range(2, 11)inertias, silhouettes = [], []rng = np.random.default_rng(RANDOM_STATE)sample_idx = rng.choice(len(X_scaled), size=min(10000, len(X_scaled)), replace=False)for k in k_range:    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)    labels = km.fit_predict(X_scaled)    inertias.append(km.inertia_)    silhouettes.append(silhouette_score(X_scaled[sample_idx], labels[sample_idx]))    print(f"k = {k:2d} | inertia = {km.inertia_:>12,.0f} | silhouette = {silhouettes[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))axes[0].plot(list(k_range), inertias, marker="o", color="#4C72B0")axes[0].set_title("(a) Elbow method — within-cluster sum of squares")axes[0].set_xlabel("Number of clusters (k)"); axes[0].set_ylabel("Inertia")axes[0].axvline(4, color="red", linestyle="--", alpha=0.6)axes[1].plot(list(k_range), silhouettes, marker="s", color="#C44E52")axes[1].set_title("(b) Silhouette score by k")axes[1].set_xlabel("Number of clusters (k)"); axes[1].set_ylabel("Silhouette score")best_k_sil = list(k_range)[int(np.argmax(silhouettes))]axes[1].axvline(best_k_sil, color="red", linestyle="--", alpha=0.6,                label=f"best k = {best_k_sil}")axes[1].legend()fig.suptitle("Figure 14. Selecting the optimal number of clusters", fontsize=13, fontweight="bold")finish_fig("fig14_elbow_silhouette")print(f"\nHighest silhouette score at k = {best_k_sil} ({max(silhouettes):.4f})")# --- Justifying the final k ---------------------------------------------------# The silhouette curve is highest at k = 2, but EVERY value is very small (below 0.03),# which means the data has no strongly separated cluster structure at any k. Because the# curve is essentially flat, the silhouette score alone cannot discriminate between the# candidates, so the elbow and the operational interpretability of the solution are used# as the deciding evidence.ELBOW_K = 4inertia_drops = np.diff(inertias)print("\nInertia reduction per additional cluster:", [int(d) for d in inertia_drops])print(f"The largest relative drop in inertia occurs up to k = {ELBOW_K}, after which the curve "      "flattens noticeably (the 'elbow').")print(f"Silhouette at k = 2 is {silhouettes[0]:.4f} and at k = {ELBOW_K} it is "      f"{silhouettes[ELBOW_K - 2]:.4f} — a negligible difference.")print(f"FINAL CHOICE: k = {ELBOW_K}. A two-cluster solution would merely split the data into a "      "large and a small group with little operational meaning, whereas four clusters are "      "supported by the elbow and produce interpretable, actionable traffic-condition profiles.")print("The uniformly low silhouette values are reported as a limitation of the clustering in "      "Section 11.3.")OPTIMAL_K = ELBOW_KRESULTS["kmeans"] = {"inertias": [float(i) for i in inertias],                     "silhouettes": [float(s) for s in silhouettes],                     "k_values": list(k_range),                     "best_k_silhouette": int(best_k_sil),                     "elbow_k": int(ELBOW_K),                     "chosen_k": int(OPTIMAL_K),                     "max_silhouette": float(max(silhouettes))}

### 10.3 Final clustering and PCA visualisation

In [ ]:
kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=25)cluster_labels = kmeans.fit_predict(X_scaled)model_df = model_df.assign(Cluster=cluster_labels)print(f"Final k-means model fitted with k = {OPTIMAL_K}")print("\nCluster sizes:")print(pd.Series(cluster_labels).value_counts().sort_index().to_string())

In [ ]:
# PCA projection for a 2-D visualisation of the clusters.pca = PCA(n_components=2, random_state=RANDOM_STATE)X_pca = pca.fit_transform(X_scaled)print("Explained variance by the two principal components: "      f"{pca.explained_variance_ratio_.sum() * 100:.2f}%")fig, axes = plt.subplots(1, 2, figsize=(17, 6.5))scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap="viridis", s=6, alpha=0.5)axes[0].set_title(f"(a) k-means clusters (k = {OPTIMAL_K}) projected on the first two PCs")axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")plt.colorbar(scatter, ax=axes[0], label="Cluster")sc = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=model_df["Severity"], cmap="rocket", s=6, alpha=0.5)axes[1].set_title("(b) The same projection coloured by the (unused) Severity label")axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")plt.colorbar(sc, ax=axes[1], label="Severity")fig.suptitle("Figure 15. Cluster structure versus the withheld severity label", fontsize=13, fontweight="bold")finish_fig("fig15_clusters_pca")

### 10.4 Cluster profiling and interpretationEach cluster is profiled by the mean of the most informative features. The profile is what gives theclusters their operational meaning — for example a cluster with high `Traffic_Density`, high`Vehicle_Count` and high `Accident_History` identifies a **high-risk, congested environment**.

In [ ]:
profile_cols = ["Traffic_Density", "Vehicle_Count", "Accident_History", "Average_Speed" if "Average_Speed" in model_df else "Speed_Limit",                "Speed_Limit", "Visibility", "Temperature", "Humidity", "Wind_Speed", "Pedestrian_Count",                "Lane_Count", "Travel_Time_Minutes"]profile_cols = [c for c in dict.fromkeys(profile_cols) if c in model_df.columns]cluster_profile = model_df.groupby("Cluster")[profile_cols].mean().round(2)cluster_profile["n_incidents"] = model_df.groupby("Cluster").size()print("Cluster profiles (mean feature values):")print(cluster_profile.to_string())RESULTS["cluster_profile"] = cluster_profile.to_dict("index")

In [ ]:
# Heat-map of the standardised cluster profile -- reveals which features define each cluster.profile_z = (cluster_profile[profile_cols] - cluster_profile[profile_cols].mean()) / \            cluster_profile[profile_cols].std(ddof=0)fig, ax = plt.subplots(figsize=(13, 4.5))sns.heatmap(profile_z, annot=cluster_profile[profile_cols].values, fmt=".1f", cmap="coolwarm",            center=0, ax=ax, cbar_kws={"label": "z-score"})ax.set_title("Figure 16. Cluster profile heat-map (colour = z-score, label = mean value)")ax.set_xlabel("Feature"); ax.set_ylabel("Cluster")finish_fig("fig16_cluster_profile_heatmap")

In [ ]:
# Relationship between the discovered clusters and the severity label.crosstab = pd.crosstab(model_df["Cluster"], model_df["Severity"])crosstab_pct = (crosstab.div(crosstab.sum(axis=1), axis=0) * 100).round(2)print("Cluster x Severity counts:")print(crosstab.to_string())print("\nCluster x Severity (row %):")print(crosstab_pct.to_string())print("\nMean severity per cluster:")print(model_df.groupby("Cluster")["Severity"].agg(["mean", "count"]).round(3).to_string())RESULTS["cluster_severity"] = crosstab_pct.to_dict("index")fig, ax = plt.subplots(figsize=(10, 5.5))crosstab_pct.plot(kind="bar", stacked=True, ax=ax, colormap="viridis")ax.set_title("Figure 17. Severity composition of each cluster (row %)")ax.set_ylabel("Percentage of incidents"); ax.set_xlabel("Cluster")ax.legend(title="Severity", bbox_to_anchor=(1.02, 1), loc="upper left")finish_fig("fig17_cluster_severity_composition")

### 10.5 Abnormal-behaviour (anomaly) detectionThe distance between each incident and its assigned cluster centroid measures how "unusual" itscombination of conditions is. Incidents in the top 1% of this distance are flagged as abnormal —these are the rare, atypical situations that a monitoring centre should investigate first.

In [ ]:
centroids = kmeans.cluster_centers_distances = np.linalg.norm(X_scaled - centroids[cluster_labels], axis=1)model_df = model_df.assign(Centroid_Distance=distances)threshold = np.quantile(distances, 0.99)model_df["Anomaly"] = (model_df["Centroid_Distance"] > threshold).astype(int)print(f"Anomaly threshold (99th percentile of centroid distance): {threshold:.3f}")print(f"Flagged anomalies: {int(model_df['Anomaly'].sum())} of {len(model_df)} "      f"({100 * model_df['Anomaly'].mean():.2f}%)")print("\nMean severity of normal vs anomalous incidents:")print(model_df.groupby("Anomaly")["Severity"].agg(["mean", "count"]).round(3).to_string())print("\nWeather composition of the anomalous group:")print(model_df[model_df["Anomaly"] == 1]["Weather_Condition"].value_counts(normalize=True).round(3).head().to_string())RESULTS["anomalies"] = int(model_df["Anomaly"].sum())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=distances, cmap="magma", s=6, alpha=0.5)anom = model_df["Anomaly"].values == 1ax.scatter(X_pca[anom, 0], X_pca[anom, 1], facecolors="none", edgecolors="cyan", s=28,           linewidths=0.7, label="Flagged anomalies (top 1%)")plt.colorbar(sc, ax=ax, label="Distance to cluster centroid")ax.set_title("Figure 18. Abnormal traffic conditions flagged by centroid distance")ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend()finish_fig("fig18_anomalies")

### 10.6 Alternative algorithm and hotspot identification

In [ ]:
# Compare k-means with agglomerative (hierarchical) clustering at the same k.agg = AgglomerativeClustering(n_clusters=OPTIMAL_K)agg_labels = agg.fit_predict(X_scaled[sample_idx])sil_agg = silhouette_score(X_scaled[sample_idx], agg_labels)sil_km = silhouette_score(X_scaled[sample_idx], cluster_labels[sample_idx])algo_compare = pd.DataFrame({"algorithm": ["k-means", "Agglomerative (Ward)"],                             "silhouette": [round(sil_km, 4), round(sil_agg, 4)]})print("Clustering-algorithm comparison at k =", OPTIMAL_K)print(algo_compare.to_string(index=False))RESULTS["algorithm_compare"] = algo_compare.to_dict("records")

In [ ]:
# Hotspot identification: which cities / states contain the most high-severity incidents?high_sev = model_df[model_df["Severity"] >= 3]hotspots = (high_sev.groupby("City")            .agg(incidents=("Severity", "size"), mean_severity=("Severity", "mean"))            .sort_values("incidents", ascending=False).head(10).round(3))print("Top-10 high-risk cities (Severity >= 3 incidents):")print(hotspots.to_string())fig, ax = plt.subplots(figsize=(11, 5.5))sns.barplot(x=hotspots.index, y="incidents", data=hotspots.reset_index(), ax=ax, palette="rocket")ax.set_title("Figure 19. High-risk locations — incidents with Severity >= 3 by city")ax.set_ylabel("Number of high-severity incidents"); ax.set_xlabel("")ax.tick_params(axis="x", rotation=30)finish_fig("fig19_hotspots")RESULTS["hotspots"] = hotspots.reset_index().to_dict("records")

## 11. Critical Evaluation and Reflection### 11.1 Supervised versus unsupervised — what each approach contributedThe two families of methods answered different questions. The **supervised** models quantified howpredictable severity is from conditions that are observable at the moment an incident is reported,and produced a ranking of influential features. The **unsupervised** model ignored the label entirelyand still recovered structure: the clusters separate on traffic-intensity variables(`Traffic_Density`, `Vehicle_Count`, `Accident_History`), which is precisely the signal the supervisedmodel also relied on. The agreement between the two independent analyses is strong evidence that thetraffic-intensity signal is genuine and not an artefact of the labels.### 11.2 Effectiveness* The tuned models clearly beat the naive majority-class baseline, and the improvement in **macro-F1**  is the meaningful part: the rare Severity-1 incidents are detected far better than a  majority-class classifier ever could.* Stratification and `class_weight="balanced"` were both necessary. Without them the models collapse  onto Severity 3 and 4 and macro-F1 falls sharply.* Removing the leaking variables (`Risk_Score`, `Emergency_Response_Time`) produced a *lower but  honest* score. Reporting the leaky score would have been meaningless in deployment because those  values do not exist when a prediction is required.### 11.3 Limitations1. **Weak and partly synthetic signal.** Most environmental and infrastructure features show almost   zero correlation with severity; the achievable macro-F1 is therefore moderate. The dataset is   described in the brief as synthetic, so the results describe this generator rather than real   traffic.2. **Adjacent-class confusion.** The normalised confusion matrix shows that most errors are between   neighbouring severity levels (e.g. 3 <-> 4), which is expected and arguably tolerable   operationally, but it caps the attainable macro-F1.3. **Severe imbalance.** Severity 1 represents only ~1.6% of the data; even with balanced weighting   its recall remains the weakest of the four classes.4. **Distance-based clustering assumptions.** k-means assumes spherical, similar-sized clusters and a   Euclidean metric; the silhouette scores are modest, indicating overlapping rather than crisp   groups. The categorical attributes could not enter the distance metric directly.5. **No temporal validation.** A random split ignores time. A production system should validate   chronologically (train on earlier years, test on later years) to detect drift.### 11.4 Practical implications* A severity classifier that is reliable for the extreme classes allows a control centre to triage  automatically and to pre-position resources.* The cluster/anomaly analysis gives a complementary *unsupervised* alarm: incidents far from any  centroid are precisely the unusual events that deserve human attention.* The hotspot analysis identifies the locations where prevention investment (signage, lighting,  speed management) would have the greatest effect.### 11.5 Recommended improvements1. Add richer behavioural and infrastructure data (real sensor feeds, weather radar, road geometry).2. Model severity ordinally (the classes are ordered) using ordinal logistic regression or a   cost-sensitive loss, which should improve the adjacent-class errors.3. Apply SMOTE / ADASYN within cross-validation folds and tune the decision threshold for the rare   class instead of relying on `class_weight` alone.4. Use SHAP values for a per-incident explanation that an operator can act on.5. Add spatial clustering (DBSCAN on latitude/longitude) to detect geographic hotspots directly, and   a time-series model for incident *occurrence* forecasting.6. Validate chronologically and monitor for drift before any deployment.

## 12. ConclusionThis project built an end-to-end machine-learning pipeline for an intelligent traffic-incidentanalysis system. The raw data contained genuine quality problems — missing values, duplicated records,inconsistent categorical labels, mis-typed numeric columns, physically impossible values and a severeclass imbalance — each of which was identified, treated and justified. A leakage audit showed that`Risk_Score` and `Emergency_Response_Time` are derived from the target and would have produced adeceptively high score, so both were removed.Four supervised classifiers were implemented on a stratified pipeline and tuned with macro-F1 as theobjective. The best model substantially outperforms the majority-class baseline, and theconfusion-matrix analysis shows that the remaining errors are concentrated between adjacent severitylevels. Feature analysis with three independent methods consistently ranked traffic-intensityvariables (`Traffic_Density`, `Vehicle_Count`, `Accident_History`) as the dominant predictors, withadverse weather contributing a smaller but real effect.Unsupervised k-means, selected with the elbow method and the silhouette score, recovered traffic-condition groups that align with the severity structure without ever seeing the label, and thecentroid-distance rule flagged the 1% most abnormal incidents. Taken together the two approachessupport the project's stated goals: predicting severity, identifying high-risk locations and detectingabnormal traffic behaviour. The main caveat is the moderate strength of the underlying signal, whichis a property of this synthetic dataset rather than of the pipeline, and which sets a realisticexpectation for accuracy in any deployment.

## ReferencesBreiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5–32.https://doi.org/10.1023/A:1010933404324Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proceedings of the 22ndACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 785–794.https://doi.org/10.1145/2939672.2939785Chawla, N. V., Bowyer, K. W., Hall, L. O., & Kegelmeyer, W. P. (2002). SMOTE: Synthetic minorityover-sampling technique. *Journal of Artificial Intelligence Research, 16*, 321–357.https://doi.org/10.1613/jair.953Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters, 27*(8), 861–874.https://doi.org/10.1016/j.patrec.2005.10.010Kaufman, L., & Rousseeuw, P. J. (1990). *Finding groups in data: An introduction to clusteranalysis*. John Wiley & Sons. https://doi.org/10.1002/9780470316801Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … VanderPlas, J.(2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*,2825–2830.Rousseeuw, P. J. (1987). Silhouettes: A graphical aid to the interpretation and validation ofcluster analysis. *Journal of Computational and Applied Mathematics, 20*, 53–65.https://doi.org/10.1016/0377-0427(87)90125-7Sokolova, M., & Lapalme, G. (2009). A systematic analysis of performance measures for classificationtasks. *Information Processing & Management, 45*(4), 427–437.https://doi.org/10.1016/j.ipm.2009.03.002World Health Organization. (2023). *Road traffic injuries*.https://www.who.int/news-room/fact-sheets/detail/road-traffic-injuries

In [ ]:
# Persist the key metrics so the report can quote exactly the numbers produced here.if FIG_DIR:    with open(os.path.join(FIG_DIR, "results.json"), "w") as fh:        json.dump(RESULTS, fh, indent=2, default=str)    print("Saved results.json to", FIG_DIR)print("\nNotebook complete.")print("Best supervised model:", RESULTS.get("best_model"))